### Feature store

Este notebook visa a criação de uma feature store para o projeto da Passos Mágicos.


- O que é a feature store?
- Benefícios
- Arquitetura
- Criação da feature store
- Monitoramento
- Versionamento
- Atualização
- Camada de segurança e governança
- Seleção de feature: cálculo de relevância de features

#### O que é a feature store?

- A integração de ML a sistemas (recomendação, fraude, personalização) que são voltados para clientes introduz novos requisitos de software que são novas para muitos times.
- Cada vez mais há necessidade de gerenciamento dos conjuntos de dados e pipelines de dados.
- Facilita:
    - Implementação de novas features
    - Automação de cálculos de features
    - Preenchimentos retroativos
    - Registros de Logs
    - Compartilhamento e reutilização de pipelines de features
    - Versionamento, Lineage, Metadados
    - Consistência entre dados de treinamento e inferência: centraliza a definição da feature, uma vez registrada, ela é servida tanto para treino quanto para inferência, sem duplicação de lógica.
    - Monitoramento do pipeline
    - Governança
        - Metadados ricos: além de schema, guarda descrição da feature, owner, data de criação, versão, estatísticas de distribuição.
        - Descoberta e reuso: engenheiros podem buscar “feature: churn_score” e ver quem já usa, em quais modelos, qual a performance.
        - Auditoria: rastrear quais features foram usadas em determinado modelo e versão, útil para compliance.
        - Monitoramento: acompanhar drift e qualidade das features em produção, não só o acesso.
    - Serving otimizado
        - Usar um feature store com serving online (ex.: Feast, Tecton, Databricks Feature Store).
        - Replicar features críticas em cache distribuído (Redis, DynamoDB) para baixa latência.
        - Integrar com sistemas de streaming (Kafka, Kinesis) para atualizar features em tempo real.

- A feature store é um sistema de dados específico para aprendizado de máquina que:
    - Executa pipelines de dados que transformam dados brutos em valores de recursos;
    - Armazena e gerencia os próprios dados de recursos; e
    - Fornece dados de recursos de forma consistente para fins de treinamento e inferência.

### Versionamento

#### Usar versões differentes da mesma feature
- Quando faz sentido:
    - Dois modelos estão em produção ao mesmo tempo, mas cada um foi treinado com uma definição diferente da mesma feature.
    - Exemplo: “média de compras nos últimos 30 dias” — um modelo usa janela de 30 dias, outro usa janela de 90 dias.
    - Nesse caso, a feature store mantém duas versões da mesma feature para garantir que cada modelo continue consistente com o que foi treinado.
    - Benefício: evita retrain imediato de modelos antigos, dá estabilidade e previsibilidade.
    - Risco: aumenta a complexidade se muitas versões coexistirem.

#### Criar uma nova feature e migrar todos os modelos
- Quando faz sentido:
    - A evolução da feature é claramente superior (melhor performance, mais robusta).
    - Há capacidade de retrain dos modelos para usar a nova definição.
    - Benefício: reduz fragmentação, simplifica governança.
    - Risco: exige retrain e validação de todos os modelos dependentes, o que pode ser caro ou demorado.



As features store atuam como um hub central para dados de feature e metadados ao longo do ciclo de vida de um projeto de aprendizado de máquina. 

As features store abstraem a lógica e o processamento usados ​​para gerar uma feature, fornecendo aos usuários uma maneira fácil e padronizada de acessar todas as features de uma empresa de forma consistente em todos os ambientes em que elas são necessárias.

Os dados em feature store são usados ​​para:

- exploração e feature engineer
- iteração, treinamento e depuração de modelos
- descoberta e compartilhamento de recursos
- disponibilização de dados para um modelo em produção para inferência
- monitoramento da integridade operacional

### Monitoring

Ao executar sistemas de produção, também é importante monitorar as métricas operacionais. As feature store rastreiam métricas operacionais relacionadas à funcionalidade principal. Tais como: 
- métricas relacionadas ao armazenamento de recursos (disponibilidade, capacidade, utilização, obsolescência)
- métricas de serviço da feature (taxa de transferência, latência, taxas de erro). 
- métricas operacionais para mecanismos externos de processamento de dados (por exemplo, taxa de sucesso de tarefas, taxa de transferência, atraso e taxa de processamento).

### Registry

O registro é uma interface central para interações do usuário com o repositório de recursos. As equipes usam o registro como um catálogo comum para explorar, desenvolver, colaborar e publicar novas definições dentro e entre as equipes.

### 1. Preparação dos Dados para o Feast (Multi-ano)

O Feast (FileSource) trabalha preferencialmente com arquivos Parquet. Vamos consolidar nossos dados refinados de 2022, 2023 e 2024 em um único arquivo.

In [6]:
import pandas as pd
import os

def get_refined_path():
    curr = os.getcwd()
    for _ in range(3):
        p = os.path.join(curr, "data", "refined")
        if os.path.exists(p):
            return p
        curr = os.path.dirname(curr)
    return None

path_refined = get_refined_path()
if not path_refined:
    raise FileNotFoundError("Diretório 'data/refined' não encontrado.")

years = ["2022", "2023", "2024"]
dfs = []

for year in years:
    file_csv = os.path.join(path_refined, f"pede_refined_{year}.csv")
    if os.path.exists(file_csv):
        print(f"Lendo: {file_csv}")
        df_year = pd.read_csv(file_csv, sep=";")
        df_year['event_timestamp'] = pd.to_datetime(f"{year}-01-01")
        if 'created_timestamp' not in df_year.columns:
            df_year['created_timestamp'] = pd.Timestamp.now()
        dfs.append(df_year)

if not dfs:
    raise ValueError("Nenhum arquivo CSV encontrado.")

df_all = pd.concat(dfs, axis=0, ignore_index=True)
file_all_parquet = os.path.join(path_refined, "pede_refined_all.parquet")

try:
    df_all.to_parquet(file_all_parquet)
    print(f"Base consolidada salva em: {file_all_parquet}")
except ImportError:
    file_all_csv = os.path.join(path_refined, "pede_refined_all_consolidated.csv")
    df_all.to_csv(file_all_csv, sep=";", index=False)
    print(f"Base salva em CSV (fallback): {file_all_csv}")

print(f"Total de registros: {len(df_all)}")

Lendo: d:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\data\refined\pede_refined_2022.csv
Lendo: d:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\data\refined\pede_refined_2023.csv
Lendo: d:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\data\refined\pede_refined_2024.csv
Base consolidada salva em: d:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\data\refined\pede_refined_all.parquet
Total de registros: 3030


### 2. Aplicando as Definições do Feast

Agora vamos aplicar as definições contidas no diretório `feature_repo`. Isso criará o registro e configurará o banco de dados online (SQLite).

In [7]:
import os
repo_path = os.path.abspath("../feature_repo") if os.path.exists("../feature_repo") else os.path.abspath("feature_repo")
!cd {repo_path} && feast apply

No project found in the repository. Using project name student_performance defined in feature_store.yaml
Applying changes for project student_performance
Updated feature view student_performance_features
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "../data/refined/pede_refined_all.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "../data/refined/pede_refined_all.parquet"
meta {
  created_timestamp {
    seconds: 1773006241
    nanos: 360961000
  }
  last_updated_timestamp {
    seconds: 1773006241
    nanos: 371635000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "../data/refined/pede_refined_all.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "../data/refined/pede_refined_all.parquet"
meta {
  created_timestamp {
    seconds:

D:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\.venv\Lib\site-packages\feast\repo_config.py:359: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


### 3. Consumo Offline (Treinamento)

O consumo offline é usado para recuperar features históricas para treinar modelos de ML. Ele resolve o problema de "point-in-time join", garantindo que não haja vazamento de dados.

In [8]:
from feast import FeatureStore
import pandas as pd
import os
from datetime import datetime

repo_path = os.path.abspath("../feature_repo") if os.path.exists("../feature_repo") else os.path.abspath("feature_repo")
store = FeatureStore(repo_path=repo_path)

entity_df = pd.DataFrame.from_dict({
    "registro_unico": ["RA-1275", "RA-1276"],
    "event_timestamp": [
        datetime(2024, 1, 1),
        datetime(2024, 1, 1)
    ]
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "student_performance_features:num_idade",
        "student_performance_features:melhor_avaliacao_score",
        "student_performance_features:flag_bolsa_estudos",
        "student_performance_features:qtd_defasagem"
    ]
).to_df()

display(training_df)

d:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\.venv\Lib\site-packages\feast\repo_config.py:359: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


,registro_unico,event_timestamp,num_idade,melhor_avaliacao_score,flag_bolsa_estudos,qtd_defasagem
0,RA-1275,2024-01-01 00:00:00+00:00,8,1,0,0
1,RA-1276,2024-01-01 00:00:00+00:00,8,1,0,0


### 4. Consumo Online (Inferência)

Para o consumo online, primeiro precisamos persistir os dados mais recentes no Online Store (Materialização).

In [9]:
!cd {repo_path} && feast materialize-incremental 2025-01-01T00:00:00

Materializing 1 feature views to 2025-01-01 00:00:00+00:00 into the sqlite online store.

student_performance_features from 2025-01-01 00:00:00+00:00 to 2025-01-01 00:00:00+00:00:


D:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\.venv\Lib\site-packages\feast\repo_config.py:359: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


In [ ]:
import pprint

online_features = store.get_online_features(
    features=[
        "student_performance_features:num_idade",
        "student_performance_features:melhor_avaliacao_score",
        "student_performance_features:num_fase_atual",
    ],
    entity_rows=[{"registro_unico": "RA-1275"}]
).to_dict()

pprint.pprint(online_features)

{'melhor_avaliacao_score': [1],
 'num_fase_atual': [0],
 'num_idade': [8],
 'registro_unico': ['RA-1275']}


d:\arquivos_antigos\Projetos\FIAP\Fase5\tech_challenger_5_project\.venv\Lib\site-packages\feast\infra\key_encoding_utils.py:141: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


: 